# 1. Extract rare SNPs from GnomAD v3.0

This document describes the steps taken using Hail (python) to extract RARE mutations with FA < 0.01 or FA < 0.001.

gnomAD v3.0 whole-genome sequencing data is publicly available on Google Cloud. The reference genome used is GRCh38.

In [1]:
import hail as hl

# Initialize Hail with the GCS connector
hl.init(spark_conf={
    'spark.jars': 'file:///home/alexpalazzo1/hail/jars/gcs-connector-hadoop3-latest.jar',
    'spark.hadoop.google.cloud.auth.service.account.enable': 'true',
    'spark.hadoop.google.cloud.auth.service.account.json.keyfile': '/home/alexpalazzo1/hail/gnomad-hail-439921-1ae1364a1b5a.json',
    'spark.hadoop.fs.gs.impl': 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem',
    'spark.hadoop.fs.AbstractFileSystem.gs.impl': 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS'
})

################################## Import data ###################################################################
# List of VCF files stored in Google Cloud for each chromosome
vcf_fil = [f"gs://gcp-public-data--gnomad/release/3.0/vcf/genomes/gnomad.genomes.r3.0.sites.chr{chrom}.vcf.bgz" 
           for chrom in list(range(1, 23)) + ['X', 'Y']]

# Load the VCF files into a single MatrixTable
mt = hl.import_vcf(vcf_fil, min_partitions=4, reference_genome='GRCh38')

print(f"Total variants loaded: {mt.count_rows()}")

################################################# Filters and Selection #############################################
# Filter SNPs only
mt = mt.filter_rows(hl.is_snp(mt.alleles[0], mt.alleles[1]))
print(f"SNPs only: {mt.count_rows()}")

# Filter high-quality variants (no filters applied)
mt_filtered = mt.filter_rows(hl.len(mt.filters) == 0)
print(f"After quality filtering: {mt_filtered.count_rows()}")

# Filter out AC0 variants (not observed)
mt_filtered = mt_filtered.filter_rows(~mt_filtered.filters.contains("AC0"))
print(f"After removing AC0 variants: {mt_filtered.count_rows()}")

# Ensure AF > 0
mt_filtered = mt_filtered.filter_rows(mt_filtered.info.AF[0] > 0.0)
print(f"With AF > 0: {mt_filtered.count_rows()}")

# Filter for AF < 0.1% (0.001), change for AF < 1%
mt_rare = mt_filtered.filter_rows(mt_filtered.info.AF[0] < 0.001)
print(f"Variants with AF < 0.1%: {mt_rare.count_rows()}")

# Annotate the MatrixTable with allele frequencies
mt_af = mt_rare.annotate_rows(af=mt_rare.info.AF[0])

# Select relevant fields
results = mt_af.select_rows(
    chromosome=mt_af.locus.contig,
    position=mt_af.locus.position,
    ref_allele=mt_af.alleles[0],
    alt_allele=mt_af.alleles[1],
    allele_frequency=mt_af.af
)

############################################## Export data ######################################################
# Export the results to a table
output_file = '/home/alexpalazzo1/Documents/Tina/Rare_SNP/gnomad_AF_less_than_0.1percent.tsv'
results_table = results.rows()
results_table.export(output_file)
print(f"Export done: {output_file}")

# Print some statistics
print(f"Total rare variants exported: {results_table.count()}")

# Optional: Show a few examples
print("First few variants:")
results_table.show(5)

# Stop Hail
hl.stop()

Loading BokehJS ...

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
Running on Apache Spark version 3.5.3
SparkUI available at http://142.1.241.113:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.133-4c60fddb171a
LOGGING: writing to /home/alexpalazzo1/Documents/Tina/Rare_SNP/hail-20250824-1757-0.2.133-4c60fddb171a.log
SLF4J: Failed to load class "org.slf4j.impl.StaticMDCBinder".
SLF4J: Defaulting to no-operation MDCAdapter implementation.
SLF4J: See http://www.slf4j.org/codes.html#no_static_mdc_binder for further details.
2025-08-24 17:57:19.410 Hail: INFO: scanning VCF for sortedness...(5 + 18) / 23]
2025-08-24 18:33:48.789 Hail: INFO: Coerced sorted VCF - no additional import work to do


Total variants loaded: 707950943


SNPs only: 602821631


After quality filtering: 526001545


...

Variants with AF < 0.1%: 497835684


2025-08-24 22:48:26.834 Hail: INFO: merging 1887 files totalling 25.2G.../ 1886]
2025-08-24 22:48:53.704 Hail: INFO: while writing:
    /home/alexpalazzo1/Documents/Tina/Rare_SNP/gnomad_AF_less_than_0.1percent.tsv
  merge time: 26.869s


Export done: /home/alexpalazzo1/Documents/Tina/Rare_SNP/gnomad_AF_less_than_0.1percent.tsv


Total rare variants exported: 497835684
First few variants:


,,,,,,
locus,alleles,chromosome,position,ref_allele,alt_allele,allele_frequency
locus<GRCh38>,array<str>,str,int32,str,str,float64
chr1:10114,"[""T"",""C""]","""chr1""",10114,"""T""","""C""",2.29e-04
chr1:10132,"[""T"",""C""]","""chr1""",10132,"""T""","""C""",2.39e-04
chr1:10134,"[""A"",""G""]","""chr1""",10134,"""A""","""G""",1.14e-05
chr1:10138,"[""T"",""C""]","""chr1""",10138,"""T""","""C""",1.85e-04
chr1:10139,"[""A"",""T""]","""chr1""",10139,"""A""","""T""",1.73e-05


---
# 2. Map rare SNPs



## 2.1 Modify the rare SNP file

Join the all_ref column (reference nucleotide) with the all_alt column (mutation nucleotide) with a ">"

In [ ]:
#joincolumns.py file
import csv

# Define input and output file names
input_file = 'FA_all_pop_snp_rare.tsv'
output_file = 'FA_all_pop_snp_rare_defined.tsv'

# Open the input file for reading and output file for writing
with open(input_file, 'r', newline='') as infile, open(output_file, 'w', newline='') as outfile:
    reader = csv.reader(infile, delimiter='\t')
    writer = csv.writer(outfile, delimiter='\t')
    
    for row in reader:
        # Join columns 3 and 4 with ">"
        row[2] = f"{row[2]}>{row[3]}"
        # Remove the original 4th column
        row.pop(3)
        # Write the modified row to the output file
        writer.writerow(row)

## 2.2 Map rare SNPs around the TSS

Now we will map the modified version of SNPs to relative positions 1kb around the TSS of human protein coding genes.

The annotation file for TSSs comes from the Fantom5 CAGE sequencing project with accurate mapping of 5' ends and gives the direction of genes.

In [2]:
import concurrent.futures
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def read_file2(file_path):
    data = {}
    try:
        with open(file_path, 'r') as file:
            for line in file:
                parts = line.strip().split()
                chr_num = parts[0].replace('chr', '')  # Remove "chr" prefix
                start = int(parts[1])
                end = int(parts[2])
                direction = parts[3]  # New column for direction

                if chr_num not in data:
                    data[chr_num] = []
                data[chr_num].append((start, end, direction))
    except Exception as e:
        logging.error(f"Error reading file2: {e}")
        raise
    return data

def process_chunk(chunk, file2_data):
    results = []
    for line in chunk:
        try:
            parts = line.strip().split()
            chr_num = parts[0].replace('chr', '')  # Remove "chr" prefix
            coord = int(parts[1])
            extra_data = parts[2:]
            matched = False
            
            if chr_num in file2_data:
                for start, end, direction in file2_data[chr_num]:
                    if direction == '+':
                        if start <= coord <= end:
                            range_position = coord - start
                            results.append(f"{range_position} {' '.join(extra_data)} {chr_num} {coord} {direction}\n")
                            matched = True
                    elif direction == '-':
                        if end <= coord <= start:
                            range_position = start - coord
                            results.append(f"{range_position} {' '.join(extra_data)} {chr_num} {coord} {direction}\n")
                            matched = True
                    else:
                        raise ValueError(f"Invalid direction found: {direction}")

        except Exception as e:
            logging.error(f"Error processing line: {line.strip()}. Error: {e}")
    return results

def process_files(file1_path, file2_path, output_file_path, chunk_size=100000):
    file2_data = read_file2(file2_path)

    try:
        with open(file1_path, 'r') as file1, open(output_file_path, 'w') as output_file:
            next(file1)  # Skip the first line (title line)
            
            with concurrent.futures.ProcessPoolExecutor() as executor:
                chunk = []
                futures = []
                for line in file1:
                    chunk.append(line)
                    if len(chunk) == chunk_size:
                        futures.append(executor.submit(process_chunk, chunk, file2_data))
                        chunk = []
                if chunk:
                    futures.append(executor.submit(process_chunk, chunk, file2_data))

                for future in concurrent.futures.as_completed(futures):
                    results = future.result()
                    output_file.writelines(results)
            
        logging.info("File processing completed successfully.")
    except Exception as e:
        logging.error(f"Error processing files: {e}")

# Paths to input files and output file
file1_path = 'veryrare/FA_all_pop_snp_veryrare_defined.tsv'
file2_path = 'Fantom5_CAGE_hg38_gencode.v40_TSS_1kb-.txt'
output_file_path = 'veryrare/raresnp_mapped_TSS_veryrare-.txt'

# Process the files
process_files(file1_path, file2_path, output_file_path)

2025-09-02 12:11:26,890 - INFO - File processing completed successfully.


## 2.3 Mapping backwards (for aligning by the end of the exon and mapping rareSNPs backwards)

In [2]:
#map_dnm.py file
import concurrent.futures
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def read_file2(file_path):
    data = {}
    try:
        with open(file_path, 'r') as file:
            for line in file:
                parts = line.strip().split()
                chr_num = parts[0].replace('chr', '')  # Remove "chr" prefix
                start = int(parts[1])
                end = int(parts[2])
                direction = parts[3]  # New column for direction

                # Store data correctly based on direction
                if chr_num not in data:
                    data[chr_num] = []
                data[chr_num].append((start, end, direction))
    except Exception as e:
        logging.error(f"Error reading file2: {e}")
        raise
    return data

def process_chunk(chunk, file2_data):
    results = []
    for line in chunk:
        try:
            parts = line.strip().split()
            chr_num = parts[0].replace('chr', '')  # Remove "chr" prefix
            coord = int(parts[1])
            extra_data = parts[2:]
            
            if chr_num in file2_data:
                for start, end, direction in file2_data[chr_num]:
                    if direction == '+':
                        if start <= coord <= end:
                            range_position = coord - end
                            results.append(f"{range_position} {' '.join(extra_data)} {chr_num} {coord}\n")
                            break
                    elif direction == '-':
                        if end <= coord <= start:
                            range_position = end - coord
                            results.append(f"{range_position} {' '.join(extra_data)} {chr_num} {coord}\n")
                            break
                    else:
                        raise ValueError(f"Invalid direction found: {direction}")
        except Exception as e:
            logging.error(f"Error processing line: {line.strip()}. Error: {e}")
    return results

def process_files(file1_path, file2_path, output_file_path, chunk_size=100000):
    file2_data = read_file2(file2_path)

    try:
        with open(file1_path, 'r') as file1, open(output_file_path, 'w') as output_file:
            next(file1)  # Skip the first line (title line)
            
            with concurrent.futures.ProcessPoolExecutor() as executor:
                chunk = []
                futures = []
                for line in file1:
                    chunk.append(line)
                    if len(chunk) == chunk_size:
                        futures.append(executor.submit(process_chunk, chunk, file2_data))
                        chunk = []
                if chunk:
                    futures.append(executor.submit(process_chunk, chunk, file2_data))

                for future in concurrent.futures.as_completed(futures):
                    results = future.result()
                    output_file.writelines(results)
            
        logging.info("File processing completed successfully.")
    except Exception as e:
        logging.error(f"Error processing files: {e}")

# Paths to input files and output file
file1_path = 'FA_all_pop_snp_rare_defined.tsv'
file2_path = 'exon_aligned/Fantom5_CAGE_hg38_gencode.v40_TSS_EIB_Updated_TSStoEIB+.txt'
output_file_path = 'exon_aligned/rareSNP_mapped_TSStoEIB+_backward.txt'

# Process the files
process_files(file1_path, file2_path, output_file_path)

2025-04-27 19:44:33,069 - INFO - File processing completed successfully.


## 2.4 Convert to reverse complement for negative strand

In [1]:
# Define the mutation conversion rules
conversion_dict = {
    "A>C": "T>G", "A>T": "T>A", "A>G": "T>C", 
    "C>T": "G>A", "C>A": "G>T", "C>G": "G>C",
    "G>A": "C>T", "G>T": "C>A", "G>C": "C>G",
    "T>A": "A>T", "T>C": "A>G", "T>G": "A>C"
}

# File paths
input_file = "around_intron/rareSNP_mapped_1kb_around_EIB-.txt"  # replace with your input file path
output_file = "around_intron/rareSNP_mapped_1kb_around_EIB-_strand_complement.txt"

# Process the file
with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        line = line.strip()  # Remove leading/trailing whitespace
        if line.endswith("+"):
            # Write the line as is if it ends with "+"
            outfile.write(line + "\n")
        elif line.endswith("-"):
            # Split the line into parts
            parts = line.split()
            # Get the mutation in the second column and apply the conversion if it exists
            mutation = parts[1]
            if mutation in conversion_dict:
                parts[1] = conversion_dict[mutation]
            # Reconstruct the line and write to output
            outfile.write(" ".join(parts) + "\n")


---
# 3. Count mapped rare SNPs

Count the sum of rare SNP across all genes mapped to each relative position around the TSS

In [10]:

import pandas as pd

# Define the path to the file
file_path = '/home/alexpalazzo1/Documents/Tina/Rare_SNP/veryrare/raresnp_mapped_TSS_notCAGE_veryrare-_strand_complement_nucleotide_surround_nonCpG_chimp_gorilla_conserved.txt'

# Read the file
with open(file_path, 'r') as file:
    lines = file.readlines()

# Initialize a dictionary to keep track of counts
mutation_types = ["A>G", "T>C", "C>G", "T>G", "C>A", "A>T", "G>C", "G>T", "C>T", "T>A", "A>C", "G>A"]
counts = {i: {mutation: 0 for mutation in mutation_types} for i in range(0, 1001)}

# Process each line
for line in lines:
    parts = line.split()
    if parts[0] == "NA":
        continue  # Skip this line
    row_index = int(parts[0])
    if 0 <= row_index <= 1000:  # Ensure the row index is within the valid range (1-1000)
        row_index -= 0  # Convert to 1-based index and add 1 for header
        mutation_type = parts[1]
        if mutation_type in mutation_types:
            counts[row_index][mutation_type] += 1

# Convert the counts dictionary to a DataFrame
df = pd.DataFrame.from_dict(counts, orient='index', columns=mutation_types)


# Save the DataFrame to a CSV file
output_file_path = '/home/alexpalazzo1/Documents/Tina/Rare_SNP/veryrare/raresnp_mapped_TSS_notCAGE_veryrare-_strand_complement_nucleotide_surround_nonCpG_chimp_gorilla_conserved_count.csv'
df.to_csv(output_file_path, index_label="Position")


### Count backwards (for aligning by the end of the exon and mapping rareSNPs backwards)

In [ ]:
#map_dnm.py file
import concurrent.futures
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def read_file2(file_path):
    data = {}
    try:
        with open(file_path, 'r') as file:
            for line in file:
                parts = line.strip().split()
                chr_num = parts[0].replace('chr', '')  # Remove "chr" prefix
                start = int(parts[1])
                end = int(parts[2])
                direction = parts[3]  # New column for direction

                # Store data correctly based on direction
                if chr_num not in data:
                    data[chr_num] = []
                data[chr_num].append((start, end, direction))
    except Exception as e:
        logging.error(f"Error reading file2: {e}")
        raise
    return data

def process_chunk(chunk, file2_data):
    results = []
    for line in chunk:
        try:
            parts = line.strip().split()
            chr_num = parts[0].replace('chr', '')  # Remove "chr" prefix
            coord = int(parts[1])
            extra_data = parts[2:]
            
            if chr_num in file2_data:
                for start, end, direction in file2_data[chr_num]:
                    if direction == '+':
                        if start <= coord <= end:
                            range_position = coord - end
                            results.append(f"{range_position} {' '.join(extra_data)} {chr_num} {coord}\n")
                            break
                    elif direction == '-':
                        if end <= coord <= start:
                            range_position = end - coord
                            results.append(f"{range_position} {' '.join(extra_data)} {chr_num} {coord}\n")
                            break
                    else:
                        raise ValueError(f"Invalid direction found: {direction}")
        except Exception as e:
            logging.error(f"Error processing line: {line.strip()}. Error: {e}")
    return results

def process_files(file1_path, file2_path, output_file_path, chunk_size=100000):
    file2_data = read_file2(file2_path)

    try:
        with open(file1_path, 'r') as file1, open(output_file_path, 'w') as output_file:
            next(file1)  # Skip the first line (title line)
            
            with concurrent.futures.ProcessPoolExecutor() as executor:
                chunk = []
                futures = []
                for line in file1:
                    chunk.append(line)
                    if len(chunk) == chunk_size:
                        futures.append(executor.submit(process_chunk, chunk, file2_data))
                        chunk = []
                if chunk:
                    futures.append(executor.submit(process_chunk, chunk, file2_data))

                for future in concurrent.futures.as_completed(futures):
                    results = future.result()
                    output_file.writelines(results)
            
        logging.info("File processing completed successfully.")
    except Exception as e:
        logging.error(f"Error processing files: {e}")

# Paths to input files and output file
file1_path = 'FA_all_pop_snp_rare_defined.tsv'
file2_path = 'exon_aligned/Fantom5_CAGE_hg38_gencode.v40_TSS_EIB_Updated_TSStoEIB+.txt'
output_file_path = 'exon_aligned/rareSNP_mapped_TSStoEIB+_backward.txt'

# Process the files
process_files(file1_path, file2_path, output_file_path)

---
# 4. Get nucleotide content


Download reference genome from Ensembl, then extract nucleotide sequence for each gene using bedtools getfasta

In [ ]:
bedtools getfasta -fi /media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa -bed /home/alexpalazzo1/Documents/Tina/Rare_SNP/veryrare/Human_Protein_Coding_Genes_GTF_Best_TSS_1kb_Seq-.gff -fo Human_Protein_Coding_Genes_GTF_Best_TSS_1kb_Seq-.fa -s -name

## Count nucleotide
Count the sum of nucleotides across all genes at each relative position around the TSS

In [5]:
def calculate_nucleotide_counts(sequences):
    sequence_length = len(sequences[0])
    
    counts = {
        'A': [0] * sequence_length,
        'T': [0] * sequence_length,
        'G': [0] * sequence_length,
        'C': [0] * sequence_length
    }
    
    for sequence in sequences:
        for i, base in enumerate(sequence):
            if base in counts:
                counts[base][i] += 1
    
    return counts

def read_fasta_and_calculate_nucleotide_counts(fasta_filename):
    sequences = []
    with open(fasta_filename, 'r') as file:
        sequence = ''
        for line in file:
            if line.startswith('>'):
                if sequence:
                    sequences.append(sequence)
                    sequence = ''
            else:
                sequence += line.strip()
        if sequence:
            sequences.append(sequence)

    if sequences:
        nucleotide_counts = calculate_nucleotide_counts(sequences)
        return nucleotide_counts
    else:
        return {}

# Example usage
fasta_filename = 'RNAPII/RNAPII_human_testes_thresholded_intergenic.fa'
nucleotide_counts = read_fasta_and_calculate_nucleotide_counts(fasta_filename)

# Output the total counts for each nucleotide at each position
for i in range(len(nucleotide_counts['A'])):
    print(f"Position {i + 1}: A={nucleotide_counts['A'][i]}, T={nucleotide_counts['T'][i]}, G={nucleotide_counts['G'][i]}, C={nucleotide_counts['C'][i]}")

# If you want to save the result to a file
output_filename = 'RNAPII/RNAPII_human_testes_thresholded_intergenic_per_position.txt'
with open(output_filename, 'w') as output_file:
    for i in range(len(nucleotide_counts['A'])):
        output_file.write(f"Position {i + 1}: A={nucleotide_counts['A'][i]}, T={nucleotide_counts['T'][i]}, G={nucleotide_counts['G'][i]}, C={nucleotide_counts['C'][i]}\n")


Position 1: A=278, T=259, G=247, C=279
Position 2: A=257, T=254, G=278, C=274
Position 3: A=259, T=245, G=281, C=278
Position 4: A=264, T=255, G=265, C=279
Position 5: A=280, T=259, G=266, C=258
Position 6: A=250, T=269, G=263, C=281
Position 7: A=235, T=269, G=268, C=291
Position 8: A=268, T=258, G=278, C=259
Position 9: A=256, T=229, G=289, C=289
Position 10: A=288, T=239, G=261, C=275
Position 11: A=270, T=243, G=274, C=276
Position 12: A=279, T=242, G=269, C=273
Position 13: A=254, T=260, G=271, C=278
Position 14: A=237, T=265, G=282, C=279
Position 15: A=244, T=275, G=264, C=280
Position 16: A=275, T=244, G=267, C=277
Position 17: A=255, T=244, G=271, C=293
Position 18: A=247, T=260, G=262, C=294
Position 19: A=240, T=248, G=284, C=291
Position 20: A=254, T=257, G=273, C=279
Position 21: A=270, T=243, G=274, C=276
Position 22: A=258, T=258, G=259, C=288
Position 23: A=268, T=256, G=266, C=273
Position 24: A=274, T=250, G=283, C=256
Position 25: A=259, T=247, G=297, C=260
Position 

### Count backwards (for aligning by the end of the exon and mapping rareSNPs backwards)

In [ ]:
def calculate_nucleotide_counts_right_aligned(sequences):
    max_length = max(len(seq) for seq in sequences)
    counts = {
        'A': [0] * max_length,
        'T': [0] * max_length,
        'G': [0] * max_length,
        'C': [0] * max_length
    }

    for seq in sequences:
        seq_length = len(seq)
        for i in range(seq_length):
            # Position from the end (0 = last nucleotide)
            pos_from_end = max_length - seq_length + i
            base = seq[i]
            if base in counts:
                counts[base][pos_from_end] += 1

    return counts

def read_fasta_and_calculate_nucleotide_counts(fasta_filename):
    sequences = []
    with open(fasta_filename, 'r') as file:
        sequence = ''
        for line in file:
            if line.startswith('>'):
                if sequence:
                    sequences.append(sequence)
                    sequence = ''
            else:
                sequence += line.strip()
        if sequence:
            sequences.append(sequence)

    if sequences:
        nucleotide_counts = calculate_nucleotide_counts_right_aligned(sequences)
        return nucleotide_counts
    else:
        return {}

# Example usage
fasta_filename = 'exon_aligned/TSStoEIB-.fa'
nucleotide_counts = read_fasta_and_calculate_nucleotide_counts(fasta_filename)

# After generating nucleotide_counts, save results:
output_filename = 'exon_aligned/TSStoEIB-_nucleotide_count_backward.txt'
with open(output_filename, 'w') as output_file:
    if nucleotide_counts:  # Check if not empty
        max_length = len(nucleotide_counts['A'])  # Redundant but correct
        output_file.write("Position\tA\tT\tC\tG\n")
        for i in range(max_length):
            position_label = i - max_length + 1  # 0 = last nucleotide
            output_file.write(f"{position_label}\t{nucleotide_counts['A'][i]}\t{nucleotide_counts['T'][i]}\t{nucleotide_counts['C'][i]}\t{nucleotide_counts['G'][i]}\n")
    else:
        output_file.write("No sequences found.\n")

---
# 5. Parse non-CpG rare SNPs

## Get coordinates of nucleotides immediately upstream and downstream of rare SNP

In [11]:
import csv

def process_file(input_file, output_file):
    with open(input_file, mode='r') as infile, open(output_file, mode='w', newline='') as outfile:
        reader = csv.reader(infile, delimiter=' ')
        writer = csv.writer(outfile, delimiter='\t')

        for row in reader:
            if len(row) < 2:
                continue  # Skip rows that don't have at least two columns

            chr_column = row[-4]
            coor_column = float(row[-3])
            

            coor_column_minus_2 = int(coor_column - 2)
            coor_column_plus_1 = int(coor_column + 1)

            writer.writerow([chr_column, coor_column_minus_2, coor_column_plus_1])


input_file = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare-_strand_complement.txt'
output_file = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare-_strand_complement_coordinates_around.txt'

process_file(input_file, output_file)

In [ ]:
bedtools getfasta -fi /media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa -bed raresnp_mapped_TSS_notCAGE_veryrare+_coordinates_around.txt -fo raresnp_mapped_TSS_notCAGE_veryrare+_3coords_around.fa -s -name

## Get nucleotides immediately upstream and downstream of rare SNP

In [14]:
def process_files(input_file1, input_file2, output_file):
    # Read input file 1
    with open(input_file1, 'r') as file1:
        lines1 = file1.readlines()
    
    # Read input file 2 and filter out lines starting with '>'
    with open(input_file2, 'r') as file2:
        lines2 = [line for line in file2 if not line.startswith(">")]
    
    # Ensure both files have the same number of lines
    if len(lines1) != len(lines2):
        raise ValueError("Input files must have the same number of lines (excluding lines starting with '>' in file 2).")
    
    # Combine the content
    combined_lines = [line1.strip() + "\t" + line2 for line1, line2 in zip(lines1, lines2)]
    
    # Write to output file
    with open(output_file, 'w') as outfile:
        outfile.writelines(combined_lines)

# Example usage
input_file1 = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare+.txt'
input_file2 = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare+_3coords_around.fa'
output_file = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare+_nucleotide_surround.txt'

process_files(input_file1, input_file2, output_file)


## Parse for CpG and non CpG

In [16]:
# Function to filter lines based on your criteria
def filter_lines(input_file, cpg_output_file, non_cpg_output_file):
    with open(input_file, 'r') as infile, \
         open(cpg_output_file, 'w') as cpg_outfile, \
         open(non_cpg_output_file, 'w') as non_cpg_outfile:
        for line in infile:
            columns = line.split()  # Split the line into columns
            mutation = columns[1]
            sequence = columns[-1]
            
            # Check conditions for CpG matches
            if (mutation == "C>T" and sequence[2] == "G") or (mutation == "G>A" and sequence[0] == "C"):
                cpg_outfile.write(line)
            else:
                non_cpg_outfile.write(line)

# Example usage
input_file = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare-_strand_complement_nucleotide_surround.txt'  # Replace with your actual file name
cpg_output_file = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare-_strand_complement_nucleotide_surround_CpG.txt'
non_cpg_output_file = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare-_strand_complement_nucleotide_surround_nonCpG.txt'

filter_lines(input_file, cpg_output_file, non_cpg_output_file)


## Count only the non CpG rare SNPs

In [18]:
import pandas as pd

# Define the path to the file
file_path = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare+_nucleotide_surround_nonCpG.txt'
3
# Read the file
with open(file_path, 'r') as file:
    lines = file.readlines()

# Initialize a dictionary to keep track of counts
mutation_types = ["A>G", "T>C", "C>G", "T>G", "C>A", "A>T", "G>C", "G>T", "C>T", "T>A", "A>C", "G>A"]
counts = {i: {mutation: 0 for mutation in mutation_types} for i in range(1, 1001)}

# Process each line
for line in lines:
    parts = line.split()
    if parts[0] == "NA":
        continue  # Skip this line
    row_index = int(parts[0])
    if 1 <= row_index <= 1000:  # Ensure the row index is within the valid range (1-999)
        row_index -= 0  # Convert to 1-based index and add 1 for header
        mutation_type = parts[1]
        if mutation_type in mutation_types:
            counts[row_index][mutation_type] += 1

# Convert the counts dictionary to a DataFrame
df = pd.DataFrame.from_dict(counts, orient='index', columns=mutation_types)


# Save the DataFrame to a CSV file
output_file_path = 'veryrare/raresnp_mapped_TSS_notCAGE_veryrare+_nucleotide_surround_nonCpG_count.csv'
df.to_csv(output_file_path, index_label="Position")

## Count nucleotide content with CpG information included

In [20]:
def calculate_nucleotide_counts(sequences):
    sequence_length = len(sequences[0])
    
    counts = {
        'A': [0] * sequence_length,
        'T': [0] * sequence_length,
        'G': [0] * sequence_length,
        'C': [0] * sequence_length,
        'C_in_CpG': [0] * sequence_length,    # C followed by G
        'G_in_CpG': [0] * sequence_length,     # G preceded by C
        'C_not_in_CpG': [0] * sequence_length, # C not followed by G
        'G_not_in_CpG': [0] * sequence_length  # G not preceded by C
    }
    
    for sequence in sequences:
        for i in range(len(sequence)):
            base = sequence[i]
            
            # Always count A and T normally
            if base == 'A':
                counts['A'][i] += 1
            elif base == 'T':
                counts['T'][i] += 1
                
            # For G: count total and whether in CpG
            elif base == 'G':
                counts['G'][i] += 1  # Total G count
                if i > 0 and sequence[i-1] == 'C':
                    counts['G_in_CpG'][i] += 1
                else:
                    counts['G_not_in_CpG'][i] += 1
                    
            # For C: count total and whether in CpG
            elif base == 'C':
                counts['C'][i] += 1  # Total C count
                if i < len(sequence)-1 and sequence[i+1] == 'G':
                    counts['C_in_CpG'][i] += 1
                else:
                    counts['C_not_in_CpG'][i] += 1
    
    return counts

def read_fasta_and_calculate_nucleotide_counts(fasta_filename):
    sequences = []
    with open(fasta_filename, 'r') as file:
        sequence = ''
        for line in file:
            if line.startswith('>'):
                if sequence:
                    sequences.append(sequence)
                    sequence = ''
            else:
                sequence += line.strip()
        if sequence:
            sequences.append(sequence)

    if sequences:
        nucleotide_counts = calculate_nucleotide_counts(sequences)
        return nucleotide_counts
    else:
        return {}

# Example usage
fasta_filename = 'veryrare/Human_Protein_Coding_Genes_GTF_Best_TSS_1kb_Seq-.fa'
nucleotide_counts = read_fasta_and_calculate_nucleotide_counts(fasta_filename)

# Output the counts
output_filename = 'veryrare/Human_Protein_Coding_Genes_GTF_Best_TSS_1kb_TSS_1kb-_per_position_CpG.txt'
with open(output_filename, 'w') as output_file:
    # Write header
    output_file.write("Position\tA Content\tT Content\tC Content\tG Content\tC_in_CpG\tG_in_CpG\tC_not_in_CpG\tG_not_in_CpG\n")
    
    sequence_length = len(nucleotide_counts['A'])
    for i in range(sequence_length):
        output_file.write(
            f"{i + 1}\t"
            f"{nucleotide_counts['A'][i]}\t"
            f"{nucleotide_counts['T'][i]}\t"
            f"{nucleotide_counts['C'][i]}\t"
            f"{nucleotide_counts['G'][i]}\t"
            f"{nucleotide_counts['C_in_CpG'][i]}\t"
            f"{nucleotide_counts['G_in_CpG'][i]}\t"
            f"{nucleotide_counts['C_not_in_CpG'][i]}\t"
            f"{nucleotide_counts['G_not_in_CpG'][i]}\n"
        )

print(f"Results saved to {output_filename}")

Results saved to veryrare/Human_Protein_Coding_Genes_GTF_Best_TSS_1kb_TSS_1kb-_per_position_CpG.txt
